# F1：FMI/FMU 模型交换 —— FMPy 实操
# F1: FMI/FMU Model Exchange — FMPy Hands-On

## 一、实验原理 | Principle

### 1.1 什么是 FMI？
**FMI**（Functional Mock-up Interface，功能模型接口）：模型交换的国际标准。
把仿真模型打包成 **FMU**（Functional Mock-up Unit，一个 `.fmu` zip 文件，
内含模型方程 + 求解器 + 描述文件），不同工具生成的 FMU 可以互相调用。

### 1.2 ME vs CS
- **ME**（Model Exchange，模型交换）：FMU 只导出方程，宿主负责求解。精度高，速度慢。
- **CS**（Co-Simulation，协同仿真）：FMU 自带求解器，宿主只管推进时间、交换数据。速度快，适合实时。

### 1.3 在 IEV CloudLab 中的定位
> **架构规则**：FMI 是**选修课**内容。车辆闭环控制不用 FMI（直接调仿真器 API 延迟更低），
> 但做**跨工具模型交换**时（如把 Dymola 的电机模型给 Simulink 用），FMI 是行业标准。

esmini 可导出 `EsminiOsiSource.fmu`（FMI 2.0），本实验演示 FMPy 的基本用法。

## 二、实验目的 | Objectives
1. 理解 FMI 标准和 FMU 文件结构
2. 掌握 FMPy 的核心 API（读取描述、实例化、单步）
3. 了解 ME 和 CS 的区别

## 三、学生任务 | Your Tasks
1. **运行**：验证 FMPy API 可用
2. **查阅**：FMPy 文档，了解 `read_model_description` 返回哪些信息
3. **思考题**：为什么车辆闭环不用 FMI？（提示：实时性、延迟）

## 四、验证内容 | Verification
- [ ] FMPy 成功导入
- [ ] 核心 API（read_model_description, FMU2Slave）可用

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `No module named 'fmpy'` | FMPy 未安装 | 检查镜像，或 `pip install fmpy` |
| FMU 加载失败 | 文件损坏或版本不匹配 | 确认 FMI 版本（1.0/2.0/3.0） |


In [ ]:
from fmpy import read_model_description, extract
from fmpy.fmi2 import FMU2Slave
import numpy as np
print('✓ FMPy 导入成功')


## 1. FMI 概念
- **ME (Model Exchange)**：导出模型，宿主求解
- **CS (Co-Simulation)**：FMU 自带求解器
- esmini 可导出 `EsminiOsiSource.fmu` (FMI 2.0)

## 2. FMPy API 演示
FMPy 可读取 FMU 的模型描述、提取、实例化、仿真。

In [ ]:
# 演示 FMPy 的核心 API（无真实 FMU 时展示接口）
import fmpy
print(f'FMPy 版本: {fmpy.__version__}')
print('\\n核心函数:')
print('  - read_model_description(fmu)  # 读取模型描述')
print('  - extract(fmu)                 # 解压 FMU')
print('  - FMU2Slave(...)               # 实例化从站')
print('  - fmu.instantiate()            # 实例化')
print('  - fmu.doStep()                 # 单步仿真')
# 验证 API 可用
assert hasattr(fmpy, 'read_model_description')
assert hasattr(fmpy.fmi2, 'FMU2Slave')
print('\\n✅ FMPy API 验证通过！')


## 🏆 挑战任务
1. 从 esmini 导出 `EsminiOsiSource.fmu`，用 FMPy 加载。
2. 对比 FMU 协同仿真与原生 Python 仿真的结果差异。